# Manual backfill — `total_click_by_user`

Sekali jalan: isi / update kolom **`total_click_by_user`** di:
- `daily_rider_mangkal_session_summary`
- `daily_rider_mangkal_session_summary_logs`

Sumber: agregat harian dari PostgreSQL **`ksj_user_click_tracker`** (DB `jiwa-ksj`).  
Join key: `rider_id` (tracker) = `rider_code` (DWH).

**Catatan:** `timestamp` di tracker dianggap sudah WIB (sama seperti ETL DAG) — tidak ditambah `+7 hours`.

**Sebelum jalan:** pastikan kolom `total_click_by_user` sudah ada di kedua tabel MySQL (`ALTER TABLE ... ADD COLUMN ...`).

Set **`DRY_RUN = True`** dulu untuk cek jumlah baris dari Postgres tanpa mengubah MySQL.

**Preset terbaru:** mode `single_date_log_batches` untuk **2026-05-05**, logs hanya **`11AM`** & **`03PM`**. Untuk backfill rentang lama (mis. sampai 4 Mei), ganti ke `BACKFILL_MODE = "range"` dan sesuaikan tanggal.

---
## 1. Konfigurasi — rentang tanggal **atau** satu tanggal + filter batch (logs)

| Mode | Kapan dipakai |
|------|----------------|
| `range` | Backfill banyak hari; **logs** = semua `batch_label` di rentang itu |
| `single_date_log_batches` | Satu `operating_date` (mis. 5 Mei); **logs** hanya batch yang kamu sebut (mis. `11AM`, `03PM`). **Summary** tetap di-update untuk tanggal itu (semua rider di staging). |

In [ ]:
# --- Pilih mode ---
BACKFILL_MODE = "range"  # "range" | "single_date_log_batches"

# Mode "range": agregat klik dari Postgres untuk rentang ini; logs = semua batch
BACKFILL_START_DATE = "2026-04-13"
BACKFILL_END_DATE = "2026-05-04"

# Mode "single_date_log_batches": contoh 5 Mei — hanya batch 11AM & 03PM di tabel *_logs
TARGET_OPERATING_DATE = "2026-05-05"
LOGS_BATCH_LABELS = ("11AM", "03PM")  # tuple kosong () = sama seperti mode range (semua batch)

DRY_RUN = True  # True = hanya pull dari Postgres + preview, tidak UPDATE MySQL

TABLE_SUMMARY = "daily_rider_mangkal_session_summary"
TABLE_LOGS = "daily_rider_mangkal_session_summary_logs"

if BACKFILL_MODE == "single_date_log_batches":
    _pg_start = _pg_end = TARGET_OPERATING_DATE
    print(f"Mode: single_date | tanggal={TARGET_OPERATING_DATE} | logs batch={LOGS_BATCH_LABELS}")
else:
    _pg_start, _pg_end = BACKFILL_START_DATE, BACKFILL_END_DATE
    print(f"Mode: range | {_pg_start} s/d {_pg_end} | logs: semua batch")

print(f"DRY_RUN = {DRY_RUN}")

---
## 2. Load config (MySQL DWH + PostgreSQL jiwa-ksj)

Notebook bisa di-run dari **root repo** (`repository/`) atau dari folder `dags/f_rider_mangkal_summary/` — path di-resolve otomatis.

In [ ]:
import importlib.util
from pathlib import Path


def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


_cwd = Path.cwd().resolve()
REPO_ROOT = None
for candidate in (_cwd, _cwd.parent, _cwd.parent.parent):
    if (candidate / "config" / "config_db_datawarehouse.py").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
        "Tidak menemukan config/. Coba 'Run All' dengan cwd = folder repository (yang berisi config/)."
    )

print(f"REPO_ROOT = {REPO_ROOT}")

config_dw = module_from_file(
    "config_dw", str(REPO_ROOT / "config" / "config_db_datawarehouse.py")
)
config_ksj = module_from_file(
    "config_ksj", str(REPO_ROOT / "config" / "config_db_ksj_link.py")
)

print("Config OK | DWH:", config_dw.db_target, "| PG db:", config_ksj.d_jiwa_ksj)

---
## 3. Extract agregat klik dari PostgreSQL

In [ ]:
import pandas as pd
import psycopg2
from IPython.display import display

pg_sql = """
SELECT
    TO_CHAR(timestamp, 'YYYY-MM-DD') AS operating_date,
    rider_id::text AS rider_code,
    COUNT(*)::bigint AS total_click_by_user
FROM ksj_user_click_tracker
WHERE rider_id IS NOT NULL
  AND timestamp::date >= %s::date
  AND timestamp::date <= %s::date
GROUP BY 1, 2
"""

pg_conn = psycopg2.connect(
    host=config_ksj.hostname,
    database=config_ksj.d_jiwa_ksj,
    user=config_ksj.username,
    password=config_ksj.password,
    port=config_ksj.port,
)

df_clicks = pd.read_sql_query(pg_sql, pg_conn, params=[_pg_start, _pg_end])
pg_conn.close()

df_clicks["rider_code"] = df_clicks["rider_code"].astype(str).str.strip()
df_clicks["operating_date"] = pd.to_datetime(
    df_clicks["operating_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

print(f"Baris agregat (date × rider): {len(df_clicks):,}")
display(df_clicks.head(10))

---
## 4. Apply ke MySQL (temp table + `UPDATE … JOIN`)

- **Summary:** baris yang match `(operating_date, rider_code)` di staging → `total_click_by_user` = agregat harian dari Postgres (untuk tanggal yang di-query).
- **Logs:** sama, kecuali jika `LOGS_BATCH_LABELS` tidak kosong (mode satu tanggal + batch): hanya baris dengan `batch_label` di daftar itu yang di-update (mis. hanya `11AM` dan `03PM`).

Catatan: angka klik dari Postgres adalah **total klik per hari per rider** di rentang tanggal query. Untuk dua batch di hari yang sama, kedua baris logs dapat **nilai yang sama** (total harian). Kalau nanti butuh klik “sampai jam 11” vs “sampai jam 15”, itu perlu query terpisah ke tracker (bukan backfill ini).

In [ ]:
import mysql.connector

if DRY_RUN:
    print("DRY_RUN aktif — skip MySQL.")
else:
    if df_clicks.empty:
        print("Tidak ada data klik di rentang tanggal — skip MySQL.")
    else:
        mysql_conn = mysql.connector.connect(
            host=config_dw.hostname,
            port=3306,
            user=config_dw.username,
            password=config_dw.password,
            database=config_dw.db_target,
        )
        cur = mysql_conn.cursor()

        cur.execute("DROP TEMPORARY TABLE IF EXISTS _backfill_profile_clicks")
        cur.execute(
            """
            CREATE TEMPORARY TABLE _backfill_profile_clicks (
                operating_date DATE NOT NULL,
                rider_code VARCHAR(50) NOT NULL,
                total_click_by_user BIGINT NOT NULL,
                PRIMARY KEY (operating_date, rider_code)
            )
            """
        )

        rows = [
            (r.operating_date, r.rider_code, int(r.total_click_by_user))
            for r in df_clicks.itertuples(index=False)
        ]
        cur.executemany(
            "INSERT INTO _backfill_profile_clicks (operating_date, rider_code, total_click_by_user) VALUES (%s, %s, %s)",
            rows,
        )

        cur.execute(
            f"""
            UPDATE {TABLE_SUMMARY} s
            INNER JOIN _backfill_profile_clicks c
              ON s.operating_date = c.operating_date AND s.rider_code = c.rider_code
            SET s.total_click_by_user = c.total_click_by_user
            """
        )
        n_summary = cur.rowcount

        if BACKFILL_MODE == "single_date_log_batches" and LOGS_BATCH_LABELS:
            batch_in = ", ".join(f"'{b}'" for b in LOGS_BATCH_LABELS)
            logs_where = f" WHERE l.batch_label IN ({batch_in})"
        else:
            logs_where = ""

        cur.execute(
            f"""
            UPDATE {TABLE_LOGS} l
            INNER JOIN _backfill_profile_clicks c
              ON l.operating_date = c.operating_date AND l.rider_code = c.rider_code
            SET l.total_click_by_user = c.total_click_by_user
            {logs_where}
            """
        )
        n_logs = cur.rowcount

        mysql_conn.commit()
        cur.close()
        mysql_conn.close()

        print(f"Updated {TABLE_SUMMARY}: {n_summary:,} row(s) matched")
        print(f"Updated {TABLE_LOGS}: {n_logs:,} row(s) matched")

---
## Selesai

Verifikasi cepat di MySQL (opsional):

```sql
-- Summary 5 Mei
SELECT operating_date, COUNT(*), SUM(total_click_by_user)
FROM daily_rider_mangkal_session_summary
WHERE operating_date = '2026-05-05'
GROUP BY 1;

-- Logs: per batch
SELECT batch_label, COUNT(*), SUM(total_click_by_user)
FROM daily_rider_mangkal_session_summary_logs
WHERE operating_date = '2026-05-05'
GROUP BY 1;
```